# AarogyaGrid AI — Model Benchmarking & Operational Risk Simulation
**Google Cloud Build with AI — Code for Community**

This notebook demonstrates:
1. **Synthetic Healthcare Time-Series Exploration** (16 PHCs, 60-day historical logs, zero PII)
2. **Chronological Train/Validation Splitting** (Zero data leakage)
3. **Model Benchmark**: Baseline 7-Day Moving Average vs Lagged Ridge Regression (MAE & RMSE)
4. **Smart Resource Redistribution**: Safe Surplus preservation & Distance Matrix optimization
5. **Simulated Federated AI**: Multi-node parameter aggregation (FedAvg)

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.data_generator import generate_synthetic_dataset
from src.ml.forecasting import DemandForecaster
from src.risk.risk_engine import RiskEngine
from src.redistribution.redistribution_engine import RedistributionEngine, calculate_haversine_distance_km
from src.federated.federated_sim import FederatedCoordinator

print('All AarogyaGrid AI libraries loaded successfully!')

## 1. Load Synthetic Time-Series Dataset

In [ ]:
dataset = generate_synthetic_dataset(days=60, seed=42)
df = dataset['timeseries_df']
print(f'Total records generated: {len(df)} across {len(dataset["phcs"])} PHCs')
df.head()

## 2. Benchmark ML Forecasting Model vs Baseline

In [ ]:
forecaster = DemandForecaster()
phc_id = 'phc-delhi-east-01'
phc_records = [r for r in dataset['daily_records'] if r['phcId'] == phc_id]

# Train and evaluate with strict chronological split (first 75% train, last 25% val)
model, metrics = forecaster.train_and_evaluate(phc_records, 'Paracetamol 500mg')
print('Evaluation Metrics (Paracetamol Demand Forecast):')
print(json.dumps(metrics, indent=2))

## 3. Safe Surplus Check & Smart Redistribution Simulation

In [ ]:
redist_engine = RedistributionEngine()

# Deficit PHC: Sector 22 Noida (critical on Insulin)
deficit_phc = [p for p in dataset['phcs'] if p['id'] == 'phc-noida-sec22'][0]
deficit_fc = forecaster.forecast('phc-noida-sec22', 'Insulin Glargine 100IU', current_stock=31, daily_usage=20.4)

# Precompute donor forecasts
donor_map = {
    p['id']: forecaster.forecast(p['id'], 'Insulin Glargine 100IU', current_stock=450, daily_usage=18.0)
    for p in dataset['phcs']
}

recommendations = redist_engine.find_transfer_recommendations(
    deficit_phc=deficit_phc,
    deficit_forecast=deficit_fc,
    candidate_donors=dataset['phcs'],
    donor_forecasts_map=donor_map
)

print(f'Found {len(recommendations)} safe redistribution options:')
for r in recommendations[:3]:
    print(f'• Transfer {r.quantity} units from {r.fromPhcName} -> {r.toPhcName} ({r.distanceKm} km, ETA: {r.estimatedTransitMinutes}m)')
    print(f'  Rationale: {r.reason}')

## 4. Multi-Node Federated Learning (FedAvg) Simulation

In [ ]:
coordinator = FederatedCoordinator()
fed_result = coordinator.run_federated_round(round_num=1)

print('Federated Averaging Round Results:')
print(f'Participating Nodes: {fed_result["participatingNodes"]}')
print(f'Aggregated Global Weights: {fed_result["aggregatedGlobalModel"]["weights"]}')
for node_eval in fed_result['nodeEvaluations']:
    print(f' - {node_eval["state"]}: Global Model MAE = {node_eval["localValidationWithGlobalModel"]["globalModelMAE"]}')